# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS42024/shailesh-flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

"""What this section does:** it turns the model's scores into a ranked list of content actions.
Editors work from the top down, and every row says *what* to do and *why*.

**How the queue is ranked.** Pages are sorted by `priority_score`, the model's estimate of how
likely a page is worth acting on. Each row has:
- an **action**: what a person should consider doing (refresh, review, monitor), and
- a **reason code**: the signal that stood out most for that page.

**How to read a row.** "Rank 3, REFRESH, STALE_CONTENT" means the model scores this page highly,
mainly because it hasn't been updated in a long time, so a person should open it and decide.

**How reason codes are made.** They name the input signal that is most unusual for that page,
weighted by how much the model relies on it. They are an explanation aid, not proof of cause.

**How well the ranking works.** On held-out pages, the top [K] had precision [X] against a base
rate of [Y], about [Z]x better than random. Numbers come from `w07_playbook_metrics.json`.

**What this does not claim.** The score is a priority hint. It does not say a refresh will raise
traffic, and nothing is changed automatically. A person reviews every row. """

import os
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

cwd = Path(os.getcwd()).resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "work").exists()), None)
search_root = repo_root or cwd
print("Working directory:", cwd, "| repo found:", bool(repo_root))

keys = ("action_queue", "baseline_action_score", "queue", "score", "predict")
csvs = [p for p in search_root.rglob("*.csv") if ".git" not in p.parts]
found = sorted([p for p in csvs if any(k in p.name.lower() for k in keys)],
               key=lambda p: next((i for i, k in enumerate(keys) if k in p.name.lower()), 99))
print("Candidate files:", [str(p) for p in found])

if not found:
    print("No score/queue CSV found. Run your w04 notebook first, or run this "
          "notebook locally from your cloned repo.")
else:
    queue = pd.read_csv(found[0])
    print("Using:", found[0].name, "| columns:", queue.columns.tolist())

    def pick(words):
        for w in words:
            for c in queue.columns:
                if w in c.lower():
                    return c
        return None

    score_col, rank_col = pick(["priority_score", "score"]), pick(["rank"])
    id_col, action_col, reason_col = pick(["url", "page", "id"]), pick(["action"]), pick(["reason"])

    if score_col is None:
        print("No score column found. Paste the columns line above and I'll map it.")
    else:
        queue = queue.sort_values(score_col, ascending=False).reset_index(drop=True)
        if rank_col is None:
            queue["rank"], rank_col = queue.index + 1, "rank"
        if action_col is None:
            queue["action"], action_col = "HUMAN_REVIEW", "action"
        if reason_col is None:
            print("No reason-code column in this file; add one before the paper uses it.")
            queue["reason_code"], reason_col = "NOT_AVAILABLE", "reason_code"

        display(queue.groupby([reason_col, action_col]).size()
                     .reset_index(name="pages").sort_values("pages", ascending=False))

        print("\nTop 10:")
        for _, r in queue.head(10).iterrows():
            print(f"#{int(r[rank_col])}: {r[id_col] if id_col else ''} | "
                  f"score {r[score_col]:.2f} | {r[action_col]} | because {r[reason_col]}")

        if repo_root:
            out_dir = repo_root / "work" / "outputs"
            out_dir.mkdir(parents=True, exist_ok=True)
            keep = [c for c in [rank_col, id_col, score_col, action_col, reason_col] if c]
            queue[keep].to_csv(out_dir / "action_queue.csv", index=False)
            print("\nSaved:", out_dir / "action_queue.csv")

        mp = (repo_root / "work" / "outputs" / "w07_playbook_metrics.json") if repo_root else None
        if mp and mp.exists():
            print("\nMetrics for the text above:", json.loads(mp.read_text()))

Working directory: /content | repo found: False
Candidate files: []
No score/queue CSV found. Run your w04 notebook first, or run this notebook locally from your cloned repo.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
"""## Intended use

**Who uses it:** a content or SEO editor, or a small content team, deciding which existing pages to look at first.

**What it is for:** ordering a review queue. The model gives each page a priority score, and a person opens the top pages and decides what to do. It helps with *where to look*, not *what to decide*.

**Decision it supports:** "Which pages should a person review for a refresh this month?"

**Status:** a practical, non-production research prototype. It is not connected to any publishing system, and it has not been tested against real outcomes from acting on the queue.

## Where it is valid

- **Data:** [describe your sample, e.g. "about N pages from the course dataset, snapshot dates A to B"].
- **Label:** the model predicts "[one-sentence definition of your label]". It does not predict "this refresh will raise traffic".
- **Evidence:** on held-out pages ([time-based / random] split), precision in the top [K] was [X] against a base rate of [Y], about [Z]x better than random (see `w07_playbook_metrics.json`).

## Where it stops being valid

- **Other sites or page types.** It was only checked on [your data's type]. Treat results elsewhere as untested.
- **Changing conditions.** If search behavior, site structure, or content habits shift, the ranking may get worse (see section 4 for triggers).
- **Correlation, not cause.** Reason codes point to the signal that stood out. They do not prove why a page is declining or that a refresh will fix it.
- **New or low-traffic pages.** Little history means noisy signals, so treat their scores with extra doubt.
- **Scores are for ordering only.** A score of 0.80 does not mean "80% likely to succeed".
- **Known gaps:** [1 to 3 real ones, e.g. no seasonality feature, no content-quality signal, one time period only].

## Out of scope

Deciding which pages to delete, judging writers or content quality, forecasting revenue, or any decision about people. The no-go list in section 3 covers this in full. """

import os
import json
import pandas as pd
from pathlib import Path

cwd = Path(os.getcwd()).resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "work").exists()), None)
print("Repo found:", bool(repo_root))

# ------------------------------------------------------------------
# 1) Evidence numbers from the metrics JSON (for "Where it is valid")
# ------------------------------------------------------------------
m = {}
if repo_root:
    mp = repo_root / "work" / "outputs" / "w07_playbook_metrics.json"
    if mp.exists():
        m = json.loads(mp.read_text())

if m:
    print("\nEvidence for the text:")
    for k, v in m.items():
        print(f"  {k}: {v}")
    p_keys = [k for k in m if k.startswith("precision_at_")]
    if p_keys and m.get("base_rate"):
        k = p_keys[0]
        print(f"\n  K = {k.split('_')[-1]} | precision = {m[k]:.3f} | "
              f"base rate = {m['base_rate']:.3f} | lift = {m[k] / m['base_rate']:.1f}x")
        if m[k] / m["base_rate"] < 1.5:
            print("  NOTE: lift is modest. Say so plainly in the text.")
    if m.get("roc_auc", 0) > 0.95:
        print("  WARNING: very high AUC. Check for leakage before claiming this.")
else:
    print("\nNo metrics JSON found. Run the section 1 cell, or copy the numbers "
          "from your pipeline report.")

# ------------------------------------------------------------------
# 2) Facts about the exported queue (for "Where it stops being valid")
# ------------------------------------------------------------------
if repo_root:
    qp = repo_root / "work" / "outputs" / "action_queue.csv"
    if qp.exists():
        q = pd.read_csv(qp)
        print(f"\nQueue size: {len(q)} pages")
        if "reason_code" in q.columns:
            share = q["reason_code"].value_counts(normalize=True).round(3)
            print("\nReason code share (a code covering most pages explains little):")
            print(share.to_string())
            if share.iloc[0] > 0.6:
                print(f"  NOTE: {share.index[0]} covers {share.iloc[0]:.0%} of pages. "
                      "Mention this limit.")
        if "priority_score" in q.columns:
            print("\nScore spread (ordering is what matters, not the values):")
            print(q["priority_score"].describe().round(3).to_string())
    else:
        print("\nNo action_queue.csv yet. Run the section 1 cell first.")

# ------------------------------------------------------------------
# 3) Limits checklist: answer from what you printed above
# ------------------------------------------------------------------
print("""
Fill these into the markdown text:
  [ ] Data: how many pages, which dates? (check your data cell)
  [ ] Label: one-sentence definition
  [ ] Split: time-based or random? (split_type above)
  [ ] Lift over base rate, stated honestly
  [ ] 1 to 3 real gaps (missing features, one time period, one site type)
""")


Repo found: False

No metrics JSON found. Run the section 1 cell, or copy the numbers from your pipeline report.

Fill these into the markdown text:
  [ ] Data: how many pages, which dates? (check your data cell)
  [ ] Label: one-sentence definition
  [ ] Split: time-based or random? (split_type above)
  [ ] Lift over base rate, stated honestly
  [ ] 1 to 3 real gaps (missing features, one time period, one site type)



## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
"""## Human review: what a person must check before acting

Every row in the queue is a suggestion. Before anyone acts on a page, the reviewer must:

1. **Open the page.** Look at it directly. Do not act on a score alone.
2. **Check the reason code against the page.** Does the stated signal (e.g. stale content, declining traffic) match what you see? If not, mark the row "reason doesn't fit" and skip it.
3. **Check the business context.** Is the page seasonal, newly launched, legally sensitive, or tied to a campaign? The model has no knowledge of these.
4. **Pick the action yourself.** The suggested action is a starting point. The reviewer may choose a different one or none.
5. **Record the decision.** Write down accepted, changed, or rejected, with a short reason. These records are how we learn whether the queue helps.

**Extra scrutiny (review slowly, or escalate):**
- pages with little history or very recent publication dates
- rows where the reason code is missing or generic (e.g. HUMAN_REVIEW)
- pages with scores near the cutoff between "review" and "monitor"
- high-traffic or high-value pages, where a mistake costs more

**Spot checks:** each cycle, review a small random sample from *below* the top of the queue as well. If those pages look just as worthy, the ranking is not adding much.

## No-go list: what must NOT be automated

- **Publishing, rewriting, or deleting any page** based on a score. A person makes and owns every content change.
- **Deleting, redirecting, or de-indexing pages** because they rank low or score badly.
- **Judging writers, teams, or editors** using these scores.
- **Treating the score as a forecast** of traffic, revenue, or the result of a refresh.
- **Bulk actions on every row of one reason code** with no page-level check.
- **Using the queue on page types or sites it wasn't tested on** (see section 2).
- **Acting on legal, medical, financial, or policy content** without review by someone qualified in that area.
- **Skipping the review step** "because the model is usually right".

**Rule of thumb:** the model may *suggest an order to look in*. Only people may *decide and act*. """

import os
import pandas as pd
from pathlib import Path
from IPython.display import display

cwd = Path(os.getcwd()).resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "work").exists()), None)
qp = (repo_root / "work" / "outputs" / "action_queue.csv") if repo_root else None

if not (qp and qp.exists()):
    print("No action_queue.csv yet. Run the section 1 cell first.")
else:
    q = pd.read_csv(qp)
    N_TOP, N_SPOT = 25, 10
    print(f"Queue: {len(q)} pages")

    # Flags for rows that need extra scrutiny
    q["extra_scrutiny"] = ""
    if "reason_code" in q.columns:
        weak = q["reason_code"].isin(["NOT_AVAILABLE", "HUMAN_REVIEW"])
        q.loc[weak, "extra_scrutiny"] += "weak reason code; "
    if "priority_score" in q.columns:
        cutoff = q["priority_score"].nlargest(min(50, len(q))).min()
        near = (q["priority_score"] - cutoff).abs() < 0.05
        q.loc[near, "extra_scrutiny"] += "near cutoff; "

    top = q.head(N_TOP).assign(sample="top")
    rest = q.iloc[N_TOP:]
    spot = rest.sample(min(N_SPOT, len(rest)), random_state=42).assign(sample="spot_check") \
           if len(rest) else rest.assign(sample="spot_check")

    review = pd.concat([top, spot], ignore_index=True)

    # Columns a human fills in
    review["page_checked (y/n)"] = ""
    review["reason_fits_page (y/n)"] = ""
    review["decision (accept/change/reject)"] = ""
    review["reviewer_note"] = ""

    out = repo_root / "work" / "outputs" / "review_sheet.csv"
    review.to_csv(out, index=False)
    print("Saved:", out)

    print(f"\nRows flagged for extra scrutiny: {(review['extra_scrutiny'] != '').sum()}")
    display(review.head(10))

# The no-go list as data, so the paper can reuse it
NO_GO = [
    "Auto-publish, rewrite, or delete pages from a score",
    "Delete, redirect, or de-index pages because they score low",
    "Judge writers or teams using these scores",
    "Treat the score as a traffic or revenue forecast",
    "Bulk-act on a whole reason code without page-level checks",
    "Use on page types or sites it was not tested on",
    "Act on legal/medical/financial content without qualified review",
    "Skip human review because the model is usually right",
]
print("\nNo-go list:")
for i, item in enumerate(NO_GO, 1):
    print(f"  {i}. {item}")


No action_queue.csv yet. Run the section 1 cell first.

No-go list:
  1. Auto-publish, rewrite, or delete pages from a score
  2. Delete, redirect, or de-index pages because they score low
  3. Judge writers or teams using these scores
  4. Treat the score as a traffic or revenue forecast
  5. Bulk-act on a whole reason code without page-level checks
  6. Use on page types or sites it was not tested on
  7. Act on legal/medical/financial content without qualified review
  8. Skip human review because the model is usually right


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [2]:
"""## What would tell us the recommendations went stale?

This is a light manual check, run on a schedule (for example monthly). It is not an automated production system.

| Signal | Trigger (starting point, to tune after the first cycle) | What to do |
|---|---|---|
| **Precision@K falls** | Lift over the base rate drops below 1.5x | Retrain, or pause using the queue |
| **Input drift** | A key feature's distribution shifts a lot (PSI above 0.2) | Investigate, likely retrain |
| **Reviewers disagree** | More than 40% of top rows marked "reject" or "reason doesn't fit" | Review the rules and reason codes |
| **Reason code mix shifts** | Any code's share moves by more than 15 points | Check for site or data changes |
| **Data or site changes** | Tracking, site structure, or data source changes | Re-validate before trusting scores |
| **Age** | More than [6] months since the last retrain | Retrain on schedule |

**Retrain when:** precision or reviewer-disagreement triggers fire, or two or more drift triggers fire together. Drift alone means "look closer", not "the model is wrong".

**Pause when:** precision falls to base-rate level. Stop using the queue for prioritization until it is fixed.

**Owner and cadence:** [role] checks these [monthly] and records the result.

**Limit:** precision can only be re-measured if acted-on pages get labeled afterward. Without recorded outcomes, reviewer feedback and drift are the only warning signs. """




import json
import os
import numpy as np
import pandas as pd
from pathlib import Path

cwd = Path(os.getcwd()).resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "work").exists()), None)
OUT = (repo_root / "work" / "outputs") if repo_root else None

TRIGGERS = {
    "min_lift_over_base": 1.5,
    "psi_investigate": 0.2,
    "max_reject_share": 0.40,
    "max_reason_share_shift": 0.15,
    "max_model_age_months": 6,
}

def psi(reference, current, bins=10):
    ref, cur = pd.Series(reference).dropna(), pd.Series(current).dropna()
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return float("nan")
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.clip(np.histogram(ref, edges)[0] / len(ref), 1e-4, None)
    c = np.clip(np.histogram(cur, edges)[0] / len(cur), 1e-4, None)
    return float(np.sum((c - r) * np.log(c / r)))

if OUT is None:
    print("Repo not found. Run this locally from your cloned repo.")
else:
    mp = OUT / "w07_playbook_metrics.json"
    if mp.exists():
        m = json.loads(mp.read_text())
        p_key = next((k for k in m if k.startswith("precision_at_")), None)
        if p_key and m.get("base_rate"):
            lift = m[p_key] / m["base_rate"]
            print(f"Baseline lift: {lift:.2f}x | flag: {lift < TRIGGERS['min_lift_over_base']}")
    else:
        print("No metrics JSON yet. Run the section 1 cell first.")

    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "w07_monitoring_triggers.json").write_text(json.dumps(TRIGGERS, indent=2))
    print("Saved:", OUT / "w07_monitoring_triggers.json")

print("""
Monthly checklist:
  [ ] Re-measure precision on newly labeled pages
  [ ] Run psi() on key features: training data vs this month's pages
  [ ] Compute the reviewer reject share from the review sheet
  [ ] Compare reason code mix with last cycle
  [ ] Log the date, the numbers, and any retrain decision
""")

Repo not found. Run this locally from your cloned repo.

Monthly checklist:
  [ ] Re-measure precision on newly labeled pages
  [ ] Run psi() on key features: training data vs this month's pages
  [ ] Compute the reviewer reject share from the review sheet
  [ ] Compare reason code mix with last cycle
  [ ] Log the date, the numbers, and any retrain decision



## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
## Exports for the paper

"""This notebook regenerates the files that the paper builds on, so re-running it from top to bottom recreates all of them.

The ranked queue is saved as `action_queue.csv` in `work/outputs/`. It lists every page with its rank, priority score, suggested action, and reason code. This file stays out of git on purpose, because it is a data file and the CI leak-guard blocks those. The notebook rebuilds it whenever it runs.

The numbers behind the paper are saved in `w07_playbook_metrics.json` in `work/outputs/`. This file is committed, and it acts as the receipt for every figure the paper quotes. The monitoring rules are saved next to it in `w07_monitoring_triggers.json`, which is also committed and feeds the recommendations section.

Three figures are saved in `work/figures/` and committed so the paper can reuse them. `fig_reason_codes.png` shows what the queue is made of, `fig_action_mix.png` shows how many pages get each suggested action, and `fig_score_distribution.png` shows how the priority scores are spread.

The rule for the paper is that every number should trace back to `w07_playbook_metrics.json`, and every figure should trace back to a file in `work/figures/`. """


import os
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image

cwd = Path(os.getcwd()).resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "work").exists()), None)
if repo_root is None:
    print("Repo not found. Run this locally from your cloned repo (work/notebooks/).")
else:
    OUT = repo_root / "work" / "outputs"
    FIG = repo_root / "work" / "figures"
    OUT.mkdir(parents=True, exist_ok=True)
    FIG.mkdir(parents=True, exist_ok=True)

    qp = OUT / "action_queue.csv"
    if not qp.exists():
        print("No action_queue.csv yet. Run the section 1 cell first, then re-run this one.")
    else:
        q = pd.read_csv(qp)

        # Re-write the queue in a clean, fixed column order (this is the file the paper uses)
        preferred = ["rank", "url", "priority_score", "action", "reason_code"]
        cols = [c for c in preferred if c in q.columns] + [c for c in q.columns if c not in preferred]
        q = q[cols].sort_values("rank" if "rank" in q.columns else cols[0]).reset_index(drop=True)
        q.to_csv(qp, index=False)

        # ---- Figure 1: reason codes
        if "reason_code" in q.columns:
            counts = q["reason_code"].value_counts().sort_values()
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.barh(counts.index, counts.values, color="#4C72B0")
            ax.set_xlabel("Pages in queue")
            ax.set_title("Reason codes in the action queue")
            fig.tight_layout()
            fig.savefig(FIG / "fig_reason_codes.png", dpi=200)
            plt.close(fig)

        # ---- Figure 2: action mix
        if "action" in q.columns:
            counts = q["action"].value_counts().sort_values()
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.barh(counts.index, counts.values, color="#55A868")
            ax.set_xlabel("Pages in queue")
            ax.set_title("Suggested actions (all need human review)")
            fig.tight_layout()
            fig.savefig(FIG / "fig_action_mix.png", dpi=200)
            plt.close(fig)

        # ---- Figure 3: score distribution
        if "priority_score" in q.columns:
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.hist(q["priority_score"], bins=30, color="#C44E52")
            ax.set_xlabel("Priority score")
            ax.set_ylabel("Pages")
            ax.set_title("Distribution of priority scores")
            fig.tight_layout()
            fig.savefig(FIG / "fig_score_distribution.png", dpi=200)
            plt.close(fig)

        # ---- Manifest: confirm what exists
        print(f"Queue: {len(q)} rows | columns: {q.columns.tolist()}\n")
        expected = [
            OUT / "action_queue.csv",
            OUT / "w07_playbook_metrics.json",
            OUT / "w07_monitoring_triggers.json",
            FIG / "fig_reason_codes.png",
            FIG / "fig_action_mix.png",
            FIG / "fig_score_distribution.png",
        ]
        for p in expected:
            status = f"OK ({p.stat().st_size:,} bytes)" if p.exists() else "MISSING"
            print(f"{status:<22} {p.relative_to(repo_root)}")

        mp = OUT / "w07_playbook_metrics.json"
        if mp.exists():
            print("\nMetrics the paper will cite:", json.loads(mp.read_text()))

        for f in ["fig_reason_codes.png", "fig_action_mix.png"]:
            if (FIG / f).exists():
                display(Image(filename=str(FIG / f)))

print("""
Before you commit:
  - action_queue.csv stays OUT of git (the CI leak-guard blocks data files)
  - COMMIT: the metrics and triggers JSONs, everything in work/figures/, and this executed notebook
""")

Repo not found. Run this locally from your cloned repo (work/notebooks/).

Before you commit:
  - action_queue.csv stays OUT of git (the CI leak-guard blocks data files)
  - COMMIT: the metrics and triggers JSONs, everything in work/figures/, and this executed notebook



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.